# 🧹 02 - 数据清洗与特征工程> 原始数据 → 分析就绪数据：处理缺失值、异常值，构建分析特征

## 1. 加载数据

In [ ]:
import syssys.path.append('..')import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom scripts.utils import load_data# 加载处理后的数据df = load_data()print(f"原始数据: {df.shape[0]} 行 × {df.shape[1]} 列")print(f"\n数据类型分布:")df.dtypes.value_counts()

## 2. 数据概览

In [ ]:
# 查看有薪资数据的球员salary_mask = df['HAS_SALARY']print(f"有薪资数据: {salary_mask.sum()} 人")print(f"无薪资数据: {(~salary_mask).sum()} 人")# 查看关键字段缺失情况key_cols = ['PLAYER_NAME', 'TEAM_ABBREVIATION', 'PTS', 'REB', 'AST', 'GP', 'MIN']print(f"\n关键字段缺失情况:")print(df[key_cols].isnull().sum())

## 3. 数据清洗

In [ ]:
# 筛选有薪资且出场次数足够（≥20场）的球员df_clean = df[df['HAS_SALARY'] & (df['GP'] >= 20)].copy()print(f"清洗前: {len(df)} 人")print(f"清洗后: {len(df_clean)} 人")print(f"剔除: {len(df) - len(df_clean)} 人（无薪资或出场不足）")# 处理缺失值numeric_cols = df_clean.select_dtypes(include=[np.number]).columnsdf_clean[numeric_cols] = df_clean[numeric_cols].fillna(0)print(f"\n缺失值处理完成，当前缺失值总数: {df_clean.isnull().sum().sum()}")

## 4. 特征工程

In [ ]:
# 构建分析用特征# 4.1 薪资水平分层df_clean['SALARY_LEVEL'] = pd.cut(    df_clean['SALARY'] / 10000,  # 转换为万美元    bins=[0, 500, 1000, 2000, 3000, 6000],    labels=['底薪 (<500万)', '中产 (500-1000万)',             '主力 (1000-2000万)', '明星 (2000-3000万)',             '超巨 (>3000万)'])# 4.2 综合贡献分（简化版）metrics = {'PTS': 0.25, 'REB': 0.15, 'AST': 0.15, 'STL': 0.075, 'BLK': 0.075}for col, w in metrics.items():    if col in df_clean.columns:        df_clean[f'{col}_NORM'] = (df_clean[col] - df_clean[col].min()) / \                                   (df_clean[col].max() - df_clean[col].min() + 1e-8)        df_clean['CONTRIBUTION'] = df_clean.get('CONTRIBUTION', 0) + df_clean[f'{col}_NORM'] * w# 4.3 薪资效率指数salary_norm = (df_clean['SALARY'] - df_clean['SALARY'].min()) / \              (df_clean['SALARY'].max() - df_clean['SALARY'].min())df_clean['EFFICIENCY'] = df_clean['CONTRIBUTION'] / (salary_norm + 0.1)# 4.4 得分效率df_clean['PTS_PER_MILLION'] = df_clean['PTS'] / (df_clean['SALARY'] / 1_000_000)print("✅ 特征工程完成")print(f"\n新增特征: SALARY_LEVEL, CONTRIBUTION, EFFICIENCY, PTS_PER_MILLION")print(f"\n效率得分分布:")df_clean['EFFICIENCY'].describe()

## 5. 数据保存

In [ ]:
# 保存清洗后的数据，供后续分析使用output_path = '../data/processed/nba_clean_2023_24.csv'df_clean.to_csv(output_path, index=False)print(f"💾 清洗后数据已保存: {output_path}")print(f"   维度: {df_clean.shape[0]} 行 × {df_clean.shape[1]} 列")

## 📋 小结清洗完成：- ✅ 去除无薪资数据和出场不足的球员- ✅ 填补缺失值- ✅ 构建核心分析特征（贡献分、效率指数、薪资分层）下一节：[03_eda.ipynb](./03_eda.ipynb) 将进行探索性数据分析和可视化。